# Week 4.2 — Clustering with PCA

**Course:** Data Science for Business (YSU)  
**Prerequisite:** [Week 4.1 — Clustering](./Week_4.1_Segmentation_with_Clustering.ipynb)

---

### Learning objectives

1. Explain why **PCA** helps when features are correlated.
2. Choose the number of components using **explained variance**.
3. Cluster in PCA space and compare to raw-feature K-Means.

### Agenda (~30 min)

| Block | Topic |
|---|---|
| 1 | Extended customer features |
| 2 | PCA fit + component interpretation |
| 3 | K-Means in PCA space |


## 1. Extended customer feature table

We continue from Week 4.1 with six behavioral features per customer.


In [ ]:
from pathlib import Path
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline


def _acq_weights(n):
    w = np.linspace(1.4, 0.7, n)
    w[0] *= 1.6
    return w / w.sum()


def make_synthetic_retail(n_customers=1800, seed=5) -> pd.DataFrame:
    """Fallback data when data/data_cleared.csv is missing."""
    rng = np.random.default_rng(seed)
    months = pd.period_range("2010-12", "2011-12", freq="M")
    first = rng.choice(months[:-1], size=n_customers, p=_acq_weights(len(months) - 1))
    rows = []
    invoice = 10000
    for i, start in enumerate(first):
        cid = 10000 + i
        quality = rng.uniform(0.15, 0.55)
        first_spend = float(rng.lognormal(3.4, 0.7))
        for m in months[months >= start]:
            age = (m - start).n
            if age == 0:
                p_buy = 1.0
            else:
                p_buy = quality * (0.72 ** max(age - 1, 0))
                p_buy *= 1.25 if m.month == 12 else 1.0
                p_buy = min(p_buy, 0.95)
            if rng.random() > p_buy:
                continue
            n_lines = int(rng.integers(1, 5))
            spend = first_spend if age == 0 else first_spend * rng.uniform(0.4, 1.3)
            for _ in range(n_lines):
                invoice += 1
                qty = int(rng.integers(1, 8))
                rows.append(
                    {
                        "InvoiceNo": invoice,
                        "InvoiceDate": m.to_timestamp()
                        + pd.Timedelta(days=int(rng.integers(0, 27))),
                        "CustomerID": cid,
                        "Quantity": qty,
                        "TotalPrice": spend / n_lines,
                    }
                )
    return pd.DataFrame(rows)


def load_transactions() -> pd.DataFrame:
    for path in [Path("data/data_cleared.csv"), Path("../data/data_cleared.csv")]:
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {len(df):,} line items from {path.resolve()}")
            return df
    print("data/data_cleared.csv not found — using synthetic retail data.")
    return make_synthetic_retail()


data = load_transactions()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["CustomerID"] = pd.to_numeric(data["CustomerID"], errors="coerce")
data = data.dropna(subset=["CustomerID", "InvoiceDate"]).copy()
data["CustomerID"] = data["CustomerID"].astype("int64")
print(
    f"{data['CustomerID'].nunique():,} customers | "
    f"{data['InvoiceDate'].min().date()} to {data['InvoiceDate'].max().date()}"
)
data.head()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [ ]:
snapshot = data["InvoiceDate"].max()

dt = data.groupby(["CustomerID", "InvoiceDate"], as_index=False).agg(
    TotalPrice=("TotalPrice", "sum"),
    Quantity=("Quantity", "sum"),
    InvoiceNo=("InvoiceNo", "count"),
)

customer_data = dt.groupby("CustomerID").agg(
    AvgQuantity=("Quantity", "mean"),
    AvgDifferentProducts=("InvoiceNo", "mean"),
    Recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    Frequency=("CustomerID", "count"),
    Monetary_Value=("TotalPrice", "mean"),
    GapBetweenOrders=("InvoiceDate", lambda d: (d.max() - d.min()).days),
)

scaler = StandardScaler()
data_stand = scaler.fit_transform(customer_data)
print("Shape:", data_stand.shape)
customer_data.head()

## 2. Principal Component Analysis (PCA)

**PCA** rotates correlated features into uncorrelated **components** that capture decreasing amounts of variance.

Rule of thumb: keep enough components for ~**80% cumulative explained variance**.


In [ ]:
pca_full = PCA()
pca_full.fit(data_stand)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", linestyle="--")
ax.axhline(0.8, color="crimson", linestyle=":", label="80% threshold")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("How many PCA components to keep?")
ax.legend()
plt.tight_layout()
plt.show()
list(enumerate(cumvar.round(3), start=1))

In [ ]:
N_COMPONENTS = 3
pca = PCA(n_components=N_COMPONENTS)
pca_results = pca.fit_transform(data_stand)

loadings = pd.DataFrame(
    pca.components_,
    columns=customer_data.columns,
    index=[f"Component {i+1}" for i in range(N_COMPONENTS)],
)
loadings.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(loadings, vmin=-1, vmax=1, cmap="RdBu", annot=True, fmt=".2f", ax=ax)
ax.set_title("PCA loadings — which original features drive each component?")
plt.tight_layout()
plt.show()

**How to read loadings:** large positive/negative values show which original features move together in that component. Component 1 often looks like an overall "engagement" axis in retail data.


## 3. K-Means in PCA space

Cluster on the 3 components instead of 6 raw features. This reduces noise from correlated inputs.


In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(pca_results)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 11), wcss, marker="o", linestyle="--")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("WCSS")
ax.set_title("Elbow plot — K-Means on PCA components")
plt.tight_layout()
plt.show()

In [ ]:
kmeans_pca = KMeans(n_clusters=4, init="k-means++", random_state=42, n_init="auto")
kmeans_pca.fit(pca_results)

pca_frame = pd.DataFrame(pca_results, columns=["Component 1", "Component 2", "Component 3"])
final_data = pd.concat([customer_data.reset_index(drop=True), pca_frame], axis=1)
final_data["Segments"] = kmeans_pca.labels_
final_data.head()

In [ ]:
profiling = final_data.groupby("Segments", as_index=False).mean(numeric_only=True)
profiling["Segment_size"] = final_data.groupby("Segments").size().values
profiling["Segment_prop"] = (profiling["Segment_size"] / profiling["Segment_size"].sum() * 100).round(1)

pca_names = {0: "promising", 1: "new_and_promising", 2: "champions", 3: "lost"}
profiling["Segments"] = profiling["Segments"].map(pca_names)
final_data["Segments"] = final_data["Segments"].map(pca_names)
profiling

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=final_data,
    x="Component 2",
    y="Component 1",
    hue="Segments",
    palette="Set2",
    alpha=0.6,
    ax=ax,
)
ax.set_title("K-Means segments in PCA space (Components 1 vs 2)")
plt.tight_layout()
plt.show()

## 4. Wrap-up

PCA + K-Means often yields **cleaner separation** in 2D plots because components are orthogonal.

| Approach | Pros | Cons |
|---|---|---|
| RFM rules | Easy to explain | Fixed bins, ignores correlations |
| K-Means on raw features | Flexible | Dominated by correlated / large-scale features |
| PCA + K-Means | Less redundancy | Components harder to explain to stakeholders |

### Practice

1. Plot Components 1 vs 3 — do clusters still separate?
2. Compare segment sizes to [Week 4.1](./Week_4.1_Segmentation_with_Clustering.ipynb). Did PCA change who is "lost"?
3. When would you **not** use PCA before clustering?
